In [1]:
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig
import os, re, math, signal, contextlib, io, sys, traceback

In [2]:
dataset = load_dataset("parquet", data_files="./data/gsm8k/main/train-00000-of-00001.parquet", split="train")
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [3]:
dataset = dataset.select(range(100)).rename_column("question", "prompt")
dataset_select = dataset.select_columns(["prompt"])
dataset_select[10]

{'prompt': 'A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hunger. Over three hundred years, it has consumed 847 people. Ships have been built larger over time, so each new ship has twice as many people as the last ship. How many people were on the ship the monster ate in the first hundred years?'}

In [4]:
# 定义测试的奖励函数
def reward_length(prompts, completions, completion_ids=None, target_len=20, **kwargs):
    """
    prompts: list[str] - 输入的prompt
    completions: list[str] - 模型生成的completion
    completion_ids: list[list[int]] - tokenizer ids (可不用)
    """
    rewards = []
    for text in completions:
        diff = abs(len(text.split()) - target_len)
        rewards.append(1.0 / (1.0 + diff))
    return rewards

def reward_keyword(prompts, completions, completion_ids=None, keyword=None, **kwargs):
    if keyword is None:
        keyword = {"I think":0.5, "hello":0.2, "thanks":0.1}
    rewards = []
    for text in completions:
        text_low = text.lower()
        reward = 0.0
        for kw, value in keyword.items():
            if kw in text_low:
                reward += value 
        rewards.append(reward)
    return rewards


In [5]:
config = GRPOConfig(
    output_dir="../model/Qwengrpo",
    learning_rate=1e-5,
    num_train_epochs=2,
    # max_steps=100,
    logging_steps=5,
    max_prompt_length=256,
    max_completion_length=128,
    steps_per_generation=2,           # (defult)1 多少部进行重新采样
    num_generations=4,                # 采样prompt多少条completion
    per_device_train_batch_size=2,    # 同时训练的batch
    gradient_accumulation_steps=1,    # 小batch进行梯度累加后在更新参数
    report_to="tensorboard",          # 写入tensorboard(默认)       
 )

In [6]:
model = AutoModelForCausalLM.from_pretrained("../model/Qwen2.5-0.5B-Instruct")

peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj",],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_peft = get_peft_model(model, peft_config)

In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_ratio = 100 * trainable_params / total_params

print(f"total parameters:{total_params}")
print(f"Trainable parameters: {trainable_params}")
print(f"Trainable parameters ratio: {trainable_ratio:.4f}%")

total parameters:494303104
Trainable parameters: 270336
Trainable parameters ratio: 0.0547%


In [8]:
model_path = "../model/Qwen2.5-0.5B-Instruct"
trainer = GRPOTrainer(
    model_peft,
    args=config,
    reward_funcs = [reward_length, reward_keyword],
    train_dataset=dataset_select,
) 

In [9]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
C:\Users\hhm18\miniconda3\envs\env_rlhf\lib\site-packages\transformers\integrations\sdpa_attention.py:83: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
C:\Users\hhm18\miniconda3\envs\env_rlhf\lib\site-packages\torch\utils\checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
5,-0.028900
10,0.028600
15,0.066000
20,-0.066300
25,0.034400
30,-0.034100
35,0.000400
40,-0.000600
45,-0.037800
50,0.038600


TrainOutput(global_step=400, training_loss=9.99311194755137e-05, metrics={'train_runtime': 3286.4242, 'train_samples_per_second': 0.061, 'train_steps_per_second': 0.122, 'total_flos': 0.0, 'train_loss': 9.99311194755137e-05})

![loss and reward](./img/train-loss-and-reward.png)